# Spotify Tracks — Exploratory Data Analysis

Runs on `data/cleaned.csv` (113,533 rows after cleaning).  
See `notebooks/overview.md` for the full narrative on data quality decisions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

DATA_PATH = Path("../data/cleaned.csv")
df = pd.read_csv(DATA_PATH, index_col=0)

## 1. Overview

In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumn dtypes:")
print(df.dtypes.to_string())
df.head()

## 2. Target Analysis

114-class classification target (`track_genre`). Dataset is designed to be balanced at ~1,000 tracks per genre.
Some genres lost rows to cleaning — the bar chart below shows the spread.

In [ ]:
counts = df["track_genre"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: overall distribution as a histogram of counts-per-genre
axes[0].hist(counts.values, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of tracks per genre", fontsize=12)
axes[0].set_xlabel("Tracks in genre")
axes[0].set_ylabel("Number of genres")
axes[0].axvline(counts.mean(), color="tomato", linestyle="--", label=f"Mean = {counts.mean():.0f}")
axes[0].legend()

# Right: bottom 10 genres (most affected by cleaning)
bottom10 = counts.tail(10)
axes[1].barh(bottom10.index, bottom10.values, color="steelblue")
axes[1].set_title("10 smallest genres (post-cleaning)", fontsize=12)
axes[1].set_xlabel("Track count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Genres: {len(counts)}")
print(f"Min: {counts.min()}  Max: {counts.max()}  Mean: {counts.mean():.1f}  Std: {counts.std():.1f}")

## 3. Missing Values

Cleaning already imputed and dropped nulls. This confirms the cleaned dataset is complete.

In [ ]:
null_counts = df.isnull().sum()

if null_counts.sum() == 0:
    print("No missing values — dataset is complete after cleaning.")
else:
    null_pct = (null_counts[null_counts > 0] / len(df) * 100).round(2)
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(
        df.isnull().T,
        cbar=False,
        ax=ax,
        yticklabels=True,
    )
    ax.set_title("Missing values heatmap (white = missing)", fontsize=12)
    ax.set_xlabel("Row index")
    plt.tight_layout()
    plt.show()
    print(null_pct)

## 4. Feature Distributions

All 10 continuous audio features. Bimodal shapes in `acousticness` and `instrumentalness`
reflect a real acoustic/electronic divide in the data — not a data quality issue.

In [ ]:
audio_features = [
    "danceability", "energy", "valence", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "loudness", "tempo", "popularity",
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(audio_features):
    axes[i].hist(df[col].dropna(), bins=50, color="steelblue", edgecolor="none", alpha=0.85)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count" if i % 5 == 0 else "")
    axes[i].tick_params(labelsize=8)

fig.suptitle("Audio Feature Distributions (cleaned data)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

Among continuous audio features. Strong positive: `energy` ↔ `loudness`.
Strong negative: `energy` ↔ `acousticness`.

In [ ]:
corr_cols = [
    "popularity", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo",
]

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
import numpy as np
mask_arr = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

# Print the top absolute correlations (excluding self)
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .abs()
    .sort_values(ascending=False)
    .head(5)
)
print("Top 5 absolute correlations:")
print(corr_pairs.to_string())

## 6. Features vs Target

Box plots across 6 contrasting genres to show how audio features separate musically
distinct styles. Genres chosen to span the full range: classical and jazz (acoustic,
quieter) vs. hip-hop and reggaeton (danceable) vs. metal and punk (high energy, loud).

In [ ]:
GENRES = ["classical", "jazz", "hip-hop", "reggaeton", "heavy-metal", "punk"]
FEATURES = ["danceability", "energy", "acousticness", "valence"]

subset = df[df["track_genre"].isin(GENRES)]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

palette = sns.color_palette("Set2", n_colors=len(GENRES))

for i, feat in enumerate(FEATURES):
    sns.boxplot(
        data=subset,
        x="track_genre",
        y=feat,
        order=GENRES,
        palette=palette,
        ax=axes[i],
        linewidth=0.8,
        fliersize=2,
    )
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel("")
    axes[i].set_ylabel(feat if i == 0 else "")
    axes[i].tick_params(axis="x", rotation=30, labelsize=9)

fig.suptitle("Audio Features Across 6 Contrasting Genres", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Dataset is well-balanced.** 114 genres × ~995 tracks each. A few genres lost rows to
  cleaning (DJ mix drops, duplicate removal) but no genre dropped below ~950 tracks.
  No resampling needed before modelling.

- **`energy` and `loudness` are strongly correlated (~0.75–0.80).** Louder tracks are
  energetic by Spotify's definition. Including both may introduce mild multicollinearity;
  worth monitoring during feature selection.

- **`acousticness` and `instrumentalness` are bimodal.** Both split sharply near 0 and 1
  rather than spreading across the full range. This reflects a real divide in the data
  (acoustic vs. electronic; vocal vs. instrumental). These features may benefit from
  binarisation rather than using the raw float in tree-based models.

- **Genre separation is clearest on `energy`, `danceability`, and `acousticness`.**
  Classical and jazz cluster at high acousticness / low energy; metal and punk cluster
  at high energy / low acousticness; hip-hop and reggaeton sit high on danceability.
  These three features are likely to carry the most weight in the classifier.

- **`valence` is surprisingly weak as a genre separator.** Happy/sad affect cuts across
  genre boundaries — jazz can be melancholic or joyful; pop can go either way. Valence
  is more useful for mood-based filtering than genre prediction.